# Prediction of Prosumers' Energy Behavior

## Loadings

### Libraries and Settings

In [ ]:
!pip install polars
!pip install catboost
!pip install optuna
!pip install statsmodels
!pip install sklearn
!pip install holidays

In [2]:
import polars as pl
print(f'Polars version {pl.__version__}')

import pandas as pd
print(f'Pandas version {pd.__version__}')

import numpy as np
print(f'Numpy version {np.__version__}')

import os
import importlib

from catboost import CatBoostRegressor as CatBoost, Pool
# Dynamically import the parent module for CatBoost and get its version
catboost = importlib.import_module('catboost')
print(f'CatBoost version {catboost.__version__}')

import optuna
print(f'Optuna version {optuna.__version__}')

from statsmodels.tsa.seasonal import MSTL
# Dynamically import the parent module and get its version
statsmodels = importlib.import_module('.'.join(MSTL.__module__.split('.')[:1]))
print(f"Statsmodels version {statsmodels.__version__}")

from sklearn.pipeline import Pipeline
from sklearn.base import TransformerMixin, BaseEstimator
from sklearn.model_selection import TimeSeriesSplit
# Dynamically import the parent module for scikit-learn and get its version
sklearn = importlib.import_module('sklearn')
print(f'SciKit-Learn version {sklearn.__version__}')

import holidays
print(f'Holidays version {holidays.__version__}')

from datetime import timedelta

import copy

Polars version 1.9.0
Pandas version 2.2.3
Numpy version 1.26.4
CatBoost version 1.2.7
Optuna version 4.2.1
Statsmodels version 0.14.4
SciKit-Learn version 1.2.2
Holidays version 0.63


In [3]:
RANDOM_STATE = 3112

### Datasets

In [4]:
# Roots to datasets in Kaggle environment
ROOT_TRAIN = "/kaggle/input/predict-energy-behavior-of-prosumers/"

# Load datasets using Polars
power_train_df = pl.read_csv(f"{ROOT_TRAIN}/train.csv")
client_train_df = pl.read_csv(f"{ROOT_TRAIN}/client.csv")
electricity_train_df = pl.read_csv(f"{ROOT_TRAIN}/electricity_prices.csv")
gas_train_df = pl.read_csv(f"{ROOT_TRAIN}/gas_prices.csv")
historical_weather_train_df = pl.read_csv(f"{ROOT_TRAIN}/historical_weather.csv")
forecast_weather_train_df = pl.read_csv(f"{ROOT_TRAIN}/forecast_weather.csv")

mapping_df = pl.read_csv(f"{ROOT_TRAIN}/weather_station_to_county_mapping.csv")

In [5]:
# Set train dictionary
train_data = {
    'power_df': power_train_df,
    'client_df': client_train_df,
    'electricity_df': electricity_train_df,
    'gas_df': gas_train_df,
    'forecast_weather_df': forecast_weather_train_df,
    'historical_weather_df': historical_weather_train_df,
    'mapping_df': mapping_df
}

## Dataset Merging

In [6]:
class DataPreparation (BaseEstimator, TransformerMixin):

    def __init__(self, mode):
        """Initialize the attributes"""

        self.mode = mode # Train, retrain or test dataset
        
    def define_data_schema(self):
        """Define schema for each DataFrame"""

        self.power_schema = {
            'row_id': pl.Int32,
            'datetime': pl.Datetime,
            'prediction_unit_id': pl.Int16,
            'is_consumption': pl.Boolean,
            'county': pl.Int8,
            'is_business': pl.Boolean,
            'product_type': pl.Int8,
            'target': pl.Float64
        }

        self.client_schema = {
            'date': pl.Date,
            'county': pl.Int8,
            'is_business': pl.Boolean,
            'product_type': pl.Int8,
            'installed_capacity': pl.Float64
        }

        self.electricity_schema = {
            'forecast_date': pl.Datetime,
            'euros_per_mwh': pl.Float64
        }

        self.gas_schema = {
            'forecast_date': pl.Datetime,
            'lowest_price_per_mwh': pl.Float64,
            'highest_price_per_mwh': pl.Float64
        }

        self.forecast_weather_schema = {
            'latitude': pl.Float32,
            'longitude': pl.Float32,
            'forecast_datetime': pl.Datetime,
            'hours_ahead': pl.Int8,
            'temperature': pl.Float32,
            'cloudcover_total': pl.Float32,
            'total_precipitation': pl.Float32,
            '10_metre_u_wind_component': pl.Float32,
            '10_metre_v_wind_component': pl.Float32,
            'direct_solar_radiation': pl.Float32,
            'surface_solar_radiation_downwards': pl.Float32
        }

        self.historical_weather_schema = {
            'datetime': pl.Datetime,
            'latitude': pl.Float32,
            'longitude': pl.Float32,
            'temperature': pl.Float32,
            'surface_pressure': pl.Float32,
            'rain': pl.Float32,
            'snowfall': pl.Float32,
            'cloudcover_total': pl.Float32,
            'windspeed_10m': pl.Float32,
            'winddirection_10m': pl.Float32,
            'direct_solar_radiation': pl.Float32,
            'diffuse_radiation': pl.Float32
        }

        self.mapping_schema = {
            'county': pl.Int8,
            'latitude': pl.Float32,
            'longitude': pl.Float32,
        }

    def check_columns(self, df, required_columns):
        """Check if all required columns exist in the DataFrame"""

        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            raise ValueError(f"Missing columns: {', '.join(missing_columns)}")
        return df

    def convert_to_schema(self, data):
        """Define the conversion logic (with column existence check)"""

        self.power_df = data['power_df']
        self.client_df = data['client_df']
        self.electricity_df = data['electricity_df']
        self.gas_df = data['gas_df']
        self.historical_weather_df = data['historical_weather_df']
        self.forecast_weather_df = data['forecast_weather_df']
        self.mapping_df = data['mapping_df']

        # Power DataFrame
        if self.mode == 'test':
            self.power_df = (
                self.power_df
                    .with_columns([
                        pl.col('prediction_datetime').alias('datetime'),
                        pl.lit(None, dtype=pl.Float64).alias('target')
                    ])
            )
        self.power_df = (
            self.check_columns(
                self.power_df, 
                ['row_id','datetime', 'prediction_unit_id', 'is_consumption', 
                'county', 'is_business', 'product_type', 'target'
                ]
            )
        )
        self.power_df = (
            self.power_df
                .with_columns([
                    (pl.col('datetime').str.strptime(pl.Datetime(time_unit="us"), '%Y-%m-%d %H:%M:%S') if self.mode=='train' else [])
                ])
                .select([col for col in self.power_schema.keys()])
                .with_columns([
                    *(pl.col(col).cast(dtype) for col, dtype in self.power_schema.items() if col != 'datetime')
                ])
        )


        # Client DataFrame
        self.client_df = (
            self.check_columns(
                self.client_df, 
                ['date', 'county', 'is_business', 
                'product_type', 'installed_capacity', 
                ]
            )
        )
        self.client_df = (
            self.client_df
                .with_columns([
                    (pl.col('date').str.strptime(pl.Date, '%Y-%m-%d') if self.mode=='train' else [])
                ])
                .select([col for col in self.client_schema.keys()])
                .with_columns([
                    *(pl.col(col).cast(dtype) for col, dtype in self.client_schema.items() if col != 'date')
                ])
        )

        # Electricity DataFrame
        self.electricity_df = (
            self.check_columns(
                self.electricity_df, 
                ['forecast_date', 'euros_per_mwh']
            )
        )
        self.electricity_df = (
            self.electricity_df
                .with_columns([
                    (pl.col('forecast_date').str.strptime(pl.Datetime(time_unit="us"), '%Y-%m-%d %H:%M:%S') if self.mode=='train' else [])
                ])
                .select(self.electricity_schema.keys())
                .with_columns([
                    *(pl.col(col).cast(dtype) for col, dtype in self.electricity_schema.items() if col != 'forecast_date')    
                ])
        )
    
        # Gas DataFrame
        self.gas_df = (
            self.check_columns(
                self.gas_df, 
                ['forecast_date', 'lowest_price_per_mwh', 'highest_price_per_mwh']
            )
        )
        self.gas_df = (
            self.gas_df
                .with_columns([
                    (pl.col('forecast_date').str.strptime(pl.Date, '%Y-%m-%d').cast(pl.Datetime)  if self.mode=='train' else [])
                ])
                .select(self.gas_schema.keys())
                .with_columns([
                    *(pl.col(col).cast(dtype) for col, dtype in self.gas_schema.items() if col != 'forecast_date')
                ])
        )

        # Forecast Weather DataFrame
        self.forecast_weather_df = (
            self.check_columns(
                self.forecast_weather_df, 
                ['latitude', 'longitude', 'forecast_datetime', 'hours_ahead', 
                'temperature', 'cloudcover_total', 'total_precipitation', 
                '10_metre_u_wind_component', '10_metre_v_wind_component', 
                'direct_solar_radiation', 'surface_solar_radiation_downwards'
                ]
            )
        )
        self.forecast_weather_df = (
            self.forecast_weather_df
                .with_columns([
                    (pl.col('forecast_datetime').str.strptime(pl.Datetime(time_unit="us"), '%Y-%m-%d %H:%M:%S') if self.mode=='train' else [])
                ])
                .select(self.forecast_weather_schema.keys())
                .with_columns([
                    *(pl.col(col).cast(dtype) for col, dtype in self.forecast_weather_schema.items() if col != 'forecast_datetime') 
                ])
        )

        # Historical Weather DataFrame
        self.historical_weather_df = (
            self.check_columns(
                self.historical_weather_df, 
                ['datetime', 'latitude', 'longitude', 
                'temperature', 'surface_pressure', 'rain', 'snowfall', 
                'cloudcover_total', 'windspeed_10m', 'winddirection_10m', 
                'direct_solar_radiation', 'diffuse_radiation'
                ]
            )
        )
        self.historical_weather_df = (
            self.historical_weather_df
                .with_columns([
                    (pl.col('datetime').str.strptime(pl.Datetime(time_unit="us"), '%Y-%m-%d %H:%M:%S') if self.mode=='train' else [])
                ])
                .select(self.historical_weather_schema.keys())
                .with_columns([
                    *(pl.col(col).cast(dtype) for col, dtype in self.historical_weather_schema.items() if col != 'datetime') 
                ])
        )

        # Mapping DataFrame
        self.mapping_df = (
            self.check_columns(
                self.mapping_df, 
                ['county', 'latitude', 'longitude']
            )
        )
        self.mapping_df = (
            self.mapping_df
                .select([col for col in self.mapping_schema.keys()])
                .with_columns([
                    pl.col(col).cast(dtype) for col, dtype in self.mapping_schema.items()
                ])
        )

        if self.mode == 'train':
            self.client_df.write_csv('client_train.csv')
            self.electricity_df.write_csv('electricity_train.csv') 
            self.gas_df.write_csv('gas_train.csv')
            self.forecast_weather_df.write_csv('forecast_weather_train.csv') 
            self.historical_weather_df.write_csv('historical_weather_train.csv')
        else:
            client_train = pl.read_csv('client_train.csv', schema_overrides=self.client_schema)
            electricity_train = pl.read_csv('electricity_train.csv', schema_overrides=self.electricity_schema)
            gas_train = pl.read_csv('gas_train.csv', schema_overrides=self.gas_schema)
            forecast_weather_train = pl.read_csv('forecast_weather_train.csv', schema_overrides=self.forecast_weather_schema)
            historical_weather_train = pl.read_csv('historical_weather_train.csv', schema_overrides=self.historical_weather_schema)
            self.client_df = (
                pl.concat([client_train, self.client_df], how='vertical')
                  .unique(subset=['date', 'county', 'is_business', 'product_type'], keep='last')
            )
            self.electricity_df = (
                pl.concat([electricity_train, self.electricity_df], how='vertical')
                  .unique(subset=['forecast_date'], keep='last')
            )
            self.gas_df = (
                pl.concat([gas_train, self.gas_df], how='vertical')
                  .unique(subset=['forecast_date'], keep='last')
            )
            self.forecast_weather_df = (
                pl.concat([forecast_weather_train, self.forecast_weather_df], how='vertical')
                  .unique(subset=['latitude', 'longitude', 'forecast_datetime', 'hours_ahead'], keep='last')
            )
            self.historical_weather_df = (
                pl.concat([historical_weather_train, self.historical_weather_df], how='vertical')
                  .unique(subset=['datetime', 'latitude', 'longitude'], keep='last')
            ) 

        self.processed_data = {
            'power_df': self.power_df,
            'client_df': self.client_df,
            'electricity_df': self.electricity_df,
            'gas_df': self.gas_df,
            'forecast_weather_df': self.forecast_weather_df,
            'historical_weather_df': self.historical_weather_df,
            'mapping_df': self.mapping_df
        }

        return self.processed_data

    def fit(self, X, y=None):
        """Fit method (required for sklearn pipeline compatibility)"""
        return self

    def transform(self, X):
        """Perform conversion anf return proccessed DataFrames"""
        
        self.define_data_schema()
        processed_data = self.convert_to_schema(X)

        return processed_data

    def fit_transform(self, X, y=None):
        self.fit(X, y=None)
        return self.transform(X)

In [7]:
class DataMerging (BaseEstimator, TransformerMixin):
    """Merge various DataFrames into one"""
        
    def __init__(self, mode):
        """Initialize the attributes"""

        self.mode = mode # Train or test dataset
    
    def merge_data(self, processed_data):
        """Form dataframe: all dataframes are alternately joined to the power_df"""

        self.power_df = processed_data['power_df']
        self.client_df = processed_data['client_df']
        self.electricity_df = processed_data['electricity_df']
        self.gas_df = processed_data['gas_df']
        self.historical_weather_df = processed_data['historical_weather_df']
        self.forecast_weather_df = processed_data['forecast_weather_df']
        self.mapping_df = processed_data['mapping_df']

        # Process power data: set power_df as the main DataFrame to be joined with, rename target
        self.df = (
            self.power_df
                .rename({'target': 'power'})
                .with_columns([
                    (pl.col('datetime').dt.truncate('1d').cast(pl.Date)).alias('date')
                ])
        )

        # Process client data: perform a join of client_df with the main DataFrame
        # (don't use eic_count as it is highly correlated with installed_capacity)
        self.df = (
            self.df
                .join(
                    self.client_df,
                    how='left',
                    on=['product_type', 'county', 'is_business', 'date']
                )
        )

        # Prepare electricity_df for later join
        self.electricity_df = (
            self.electricity_df
                .with_columns([
                    # Rename 'forecast_date' to 'datetime'
                    pl.col('forecast_date').alias('datetime'),
                    # Rename and take absolute value of 'euros_per_mwh'
                    pl.col('euros_per_mwh').abs().alias('electricity_price')
                ])
        )
        
        # Process electricity data: perform a join of electricity_df with the main DataFrame
        self.df = (
            self.df
                .join(
                    self.electricity_df
                        .select(['electricity_price', 'datetime']),
                    how='left',
                    on=['datetime']
                )
        )

        # Prepare gas_df for later join    
        self.gas_df = (
            self.gas_df
                .with_columns([
                    # Rename 'forecast_date' to 'date_only'
                    pl.col('forecast_date').alias('date_only'),
                    # Compute the average gas price per MWh
                    ((pl.col('lowest_price_per_mwh') + pl.col('highest_price_per_mwh')) / 2).alias('gas_price')
                ])
        )
        # Create a DataFrame of hours from 0 to 23
        hours_df = pl.DataFrame({'hour': range(24)})
        # Cross join to replicate gas_df rows for each hour
        self.gas_df = (
            self.gas_df
                .join(hours_df, how='cross')
                .with_columns([
                     # Combine date and hour
                    (pl.col('date_only') + pl.col('hour') * pl.duration(hours=1)).alias('datetime'),
                ])
                # Select relevant columns
                .select(['datetime', 'gas_price'])  
        )

        # Process gas data: perform a join of gas_df with the main DataFrame
        self.df = (
            self.df
                .join(
                    self.gas_df,
                    how='left',
                    on=['datetime']
                )
        )


        forecast_weather_cols = [
            'temperature', 'cloudcover_total', 'total_precipitation', 
            '10_metre_u_wind_component', '10_metre_v_wind_component', 
            'direct_solar_radiation', 'surface_solar_radiation_downwards'
        ]
        # Perform a join of forecast_weather_df and mapping_df
        self.forecast_weather_df = (
            self.forecast_weather_df
                # Filter out forecast data that is more than 24 hours ahead
                .filter(pl.col('hours_ahead') <= 24)
                .with_columns(
                    # Rename 'forecast_datetime' to 'datetime'
                    pl.col('forecast_datetime').alias('datetime')
                )
                # Join with mapping_df in order to get county id
                .join(
                    self.mapping_df,
                    how='left',
                    on=['longitude', 'latitude']
                )
        )
        # Prepare forecast_weather_df for later join
        self.forecast_weather_df = (
            self.forecast_weather_df
                .group_by(['county', 'datetime'])
                # Compute mean for each forecast weather column
                .agg([pl.mean(col).alias(f'fw_{col}') for col in forecast_weather_cols])  
                .with_columns([
                    # Cap cloud cover at 100
                    pl.when(pl.col('fw_cloudcover_total') * 100 > 100)
                        .then(100)
                        .otherwise(pl.col('fw_cloudcover_total') * 100),
                    # Take absolute value of 'fw_direct_solar_radiation'  
                    pl.col('fw_direct_solar_radiation').abs(),
                    # Compute diffuse solar radiation
                    (pl.when((pl.col('fw_surface_solar_radiation_downwards').abs() - pl.col('fw_direct_solar_radiation').abs()) < 0)
                        .then(0)
                        .otherwise(pl.col('fw_surface_solar_radiation_downwards').abs() - pl.col('fw_direct_solar_radiation').abs()))
                        .alias('fw_diffuse_solar_radiation'),
                    # Scale precipitation in mm  
                    (pl.col('fw_total_precipitation') * 1000).abs()
                ])
        ) 
        # Process forecast weather data: perform a join of forecast_weather_df with the main DataFrame
        # (don't use dewpoint, cloudcover_high, cloudcover_low, cloudcover_mid, snowfall 
        # due to high correlation with other variables)
        self.df = (
            self.df
                .join(
                    self.forecast_weather_df, 
                    how='left',
                    on=['county', 'datetime']
                )
                .with_columns(
                    # Fill missing values of forecast_weather columns 
                    # (exist due to existance of unknown 12 county in client_df) with the corresponding median values
                    [pl.col(f'fw_{col}').fill_null(pl.col(f'fw_{col}').median().over('datetime')) for col in forecast_weather_cols] + 
                    [pl.col('fw_diffuse_solar_radiation').fill_null(pl.col('fw_diffuse_solar_radiation').median().over('datetime'))]
                )  
        ) 

        historical_weather_cols = [
            'temperature', 'surface_pressure', 'rain', 'snowfall', 
            'cloudcover_total', 'windspeed_10m', 'winddirection_10m', 
            'direct_solar_radiation', 'diffuse_radiation'
        ]
        # Perform a join of historical_weather_df and mapping_df
        self.historical_weather_df = (
            self.historical_weather_df
                # Join with mapping_df in order to get county id
                .join(
                    self.mapping_df,
                    how='left',
                    on=['longitude', 'latitude']
                )
        )
        # Prepare historical_weather_df for later join
        self.historical_weather_df = (
            self.historical_weather_df
                .group_by(['county', 'datetime'])
                # Compute mean for historical weather columns
                .agg([pl.mean(col).alias(f'hw_{col}') for col in historical_weather_cols])  
                .with_columns([
                    # Combine rain and snowfall into 'hw_total_precipitation'
                    (pl.col('hw_rain') + pl.col('hw_snowfall')).alias('hw_total_precipitation')
                ])
        )
        # Process historical weather data: perform a join of historicl_weather_df with the main DataFrame
        # (don't use dewpoint, rain, snowfall, cloudcover_high, cloudcover_low, cloudcover_mid, shortwave_radiation 
        # due to high correlation with other variables)
        self.df = (
            self.df
                .join(
                    self.historical_weather_df,
                    how='left',
                    on=['county', 'datetime']
                )
                .with_columns(
                    # Fill missing values of historical_weather columns 
                    # (exist due to existance of unknown 12 county in client_df) with the corresponding median values 
                    [pl.col(f'hw_{col}').fill_null(pl.col(f'hw_{col}').median().over('datetime')) for col in historical_weather_cols] +
                    [pl.col('hw_total_precipitation').fill_null(pl.col('hw_total_precipitation').median().over('datetime'))]
                ) 
        )
  
        # Generate capacity_factor feature, define the order of columns 
        self.df = (
            self.df
                .with_columns([
                    (pl.col('power') / pl.col('installed_capacity')).alias('capacity_factor')
                ])
                .select('row_id', 'datetime', 'prediction_unit_id', 'is_consumption', 
                        'county', 'is_business', 'product_type',
                        'capacity_factor', 'power', 'installed_capacity', 
                        'electricity_price', 'gas_price',
                        'fw_temperature', 'fw_cloudcover_total', 
                        'fw_total_precipitation', 
                        'fw_10_metre_u_wind_component', 'fw_10_metre_v_wind_component',
                        'fw_direct_solar_radiation', 'fw_diffuse_solar_radiation',
                        'hw_temperature', 'hw_surface_pressure',
                        'hw_cloudcover_total', 'hw_total_precipitation',
                        'hw_windspeed_10m', 'hw_winddirection_10m',
                        'hw_direct_solar_radiation', 'hw_diffuse_radiation'
                ) 
                .sort('datetime')
        )

        return self.df
    
    def fit(self, X, y=None):
        """Fit method (required for sklearn pipeline compatibility)"""
        return self

    def transform(self, X):
        """Merge data and return the DataFrame"""
        
        df = self.merge_data(X)

        return df

    def fit_transform(self, X, y=None):
        self.fit(X)
        return self.transform(X)

In [ ]:
class FeatureEngeneering(BaseEstimator, TransformerMixin):
    """Generate features"""

    def __init__(self, mode, n_lag=[12, 24, 168, 720], train_df=None, retrain_df=None):
        """Initialize the attributes"""
        
        self.mode = mode # Train, retrain or test dataset
        self.n_lag = n_lag  # List of lag intervals, assuming possible hourly, daily, weekly, monthly seasonality
        self.min_test_datetime = None
        self.min_retrain_datetime = None
        self.train_df_pl = pl.from_pandas(train_df) if train_df is not None else None
        self.retrain_df_pl = pl.from_pandas(retrain_df) if retrain_df is not None else None
        self.estonian_holidays = list(
            holidays.country_holidays("EE", years=range(2021, 2026)).keys()
        )
        self.day_before_holidays = (
            pl.Series("day_before_holidays", self.estonian_holidays)
            .cast(pl.Date) 
            - pl.duration(days=1)
        )
        # List of March (spring forward) and October (fall back) DST shift dates for 2021-2025
        self.march_dsts = (
            pl.Series("march_dsts", [
                "2021-03-28 03:00:00",
                "2022-03-27 03:00:00",
                "2023-03-26 03:00:00",
                "2024-03-31 03:00:00",
                "2025-03-30 03:00:00"
            ])
            .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S").cast(pl.Datetime("us"))
        )

        self.october_dsts = (
            pl.Series("october_dsts", [
                "2021-10-31 03:00:00",
                "2022-10-30 03:00:00",
                "2023-10-29 03:00:00",
                "2024-10-27 03:00:00",
                "2025-10-26 03:00:00"
            ])
            .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S").cast(pl.Datetime("us"))
        )

    def is_holiday(self, date):
        """Check wheither a day is a holiday"""
        return date.is_in(self.estonian_holidays)
    
    def is_day_before_holiday(self, date):
        """Check if the day is the day before a holiday"""
        return date.is_in(self.day_before_holidays)
    
    def is_spring_forward(self, date):
        """Check if the datetime is a spring-forward (March DST shift)"""
        return date.is_in(self.march_dsts)

    def is_fall_backward(self, date):
        """Check if the datetime is a fall-backward (October DST shift)"""
        return date.is_in(self.october_dsts)

    def add_datetime_features(self, df):
        """Add cyclical features"""

        df = (
            df
                # Add datetime features
                .with_columns([
                    pl.col('datetime').dt.year().alias('year'),
                    pl.col('datetime').dt.month().alias('month'),
                    pl.col('datetime').dt.day().alias('day'),
                    pl.col('datetime').dt.weekday().alias('day_of_week'),
                    pl.col('datetime').dt.hour().alias('hour'),
                    pl.col('datetime').dt.date().map_batches(self.is_holiday).alias('is_holiday'),
                    pl.col('datetime').dt.date().map_batches(self.is_day_before_holiday).alias('is_day_before_holiday'),
                    pl.col('datetime').map_batches(self.is_spring_forward).alias('is_dst_spring_forward'),
                    pl.col('datetime').map_batches(self.is_fall_backward).alias('is_dst_fall_backward')
                ])
                # Add cyclical features
                .with_columns([
                    (2 * np.pi * pl.col('hour') / 24).sin().alias('hour_sin'),
                    (2 * np.pi * pl.col('hour') / 24).cos().alias('hour_cos'),
                    (2 * np.pi * pl.col('day') / 31).sin().alias('day_sin'),
                    (2 * np.pi * pl.col('day') / 31).cos().alias('day_cos'),
                    (2 * np.pi * pl.col('day_of_week') / 7).sin().alias('day_of_week_sin'),
                    (2 * np.pi * pl.col('day_of_week') / 7).cos().alias('day_of_week_cos'),
                    (2 * np.pi * pl.col('month') / 12).sin().alias('month_sin'),
                    (2 * np.pi * pl.col('month') / 12).cos().alias('month_cos'),
                    (2 * np.pi * pl.col('year') / 2023).sin().alias('year_sin'),
                    (2 * np.pi * pl.col('year') / 2023).cos().alias('year_cos')
                ])
                # Drop unnecessary features
                .drop(['hour', 'day', 'day_of_week', 'month', 'year'])
        )

        return df
    
    def merge_data(self, df: pl.DataFrame) -> pl.DataFrame:
        """Merge data in order to calculate lag features"""
        
        # Initialise min datetime for retrain and test modes
        self.min_retrain_datetime = df['datetime'].min() if self.mode == 'retrain' else None
        self.min_test_datetime = df['datetime'].min() if self.mode == 'test' else None

        # For retraining, merge revealed data with train dataset
        if self.mode == 'retrain' and self.train_df_pl is not None:
            # Ensure both DataFrames contain all columns from the train_df, filling missing ones with None
            missing_columns = set(self.train_df_pl.columns) - set(df.columns)
            df = df.with_columns([pl.lit(None).alias(col) for col in missing_columns])
            # Ensure the order of columns in df matches the order in train_df
            df = df.select(self.train_df_pl.columns)
            # Concatenate dataframes (train dataset with retrain dataset)
            df = (pl.concat([self.train_df_pl, df], how='vertical')
                    .unique(subset=['datetime', 'prediction_unit_id', 'is_consumption',  
                                    'county', 'is_business', 'product_type'], keep='last'
                    )
                    .sort('datetime', 'prediction_unit_id', 'is_consumption', 'product_type')
            )

        # For testing, merge test dataset with train and revealed data
        if self.mode == 'test':
            # Check if both train and retrain datasets exist
            if self.train_df_pl is not None and self.retrain_df_pl is not None:
                # Merge train and retrain first (if both exist)
                merged_train = pl.concat([self.train_df_pl, self.retrain_df_pl], how='vertical')
            elif self.train_df_pl is not None:
                merged_train = self.train_df_pl
            elif self.retrain_df_pl is not None:
                merged_train = self.retrain_df_pl
            else:
                # No data to merge
                merged_train = None  
            # Ensure the columns of merged_train and df match the columns from the train DataFrame
            missing_columns = set(self.train_df_pl.columns) - set(df.columns)
            # Add missing columns to df, filling with NaNs (None in Polars)
            df = df.with_columns([pl.lit(None).alias(col) for col in missing_columns])
            # Ensure the order of columns in df matches the order in train_df
            df = df.select(self.train_df_pl.columns)
            # Concatenate the resulting dataset with df (test dataset)
            df = (pl.concat([merged_train, df], how='vertical')
                    # Drop duplicates based on key columns, keeping the latest values
                    .unique(subset=['datetime', 'prediction_unit_id', 'is_consumption',  
                                    'county', 'is_business', 'product_type'], keep='last'
                    )
                    .sort('datetime', 'prediction_unit_id', 'is_consumption', 'product_type')
            )
        
        return df
    
    def handle_missing_values(self, df: pl.DataFrame) -> pl.DataFrame:
        """Deal with missing values"""

        if self.mode != 'test':
            # Fill missing values of power with 0 if the datetime is a spring-forward date or autumn-backward date
            df = df.with_columns(
                pl.when(((pl.col('is_dst_spring_forward') == True) | (pl.col('is_dst_fall_backward') == True)) & (pl.col('power').is_null()))
                .then(0)
                .otherwise(pl.col('power'))
                .alias('power')
            )

        if self.mode == 'train':
            df = (
                df
                    # Drop missing values of power if exsist
                    .drop_nulls(['power'])
                    # Filter out specific date with huge proportion of missing values
                    .filter(~(pl.col('datetime') 
                            .is_between(pl.datetime(2021, 9, 1, 0, 0, 0), pl.datetime(2021, 9, 2, 23, 59, 59)))
                           )
            )

        forecast_weather_cols =  [
            'temperature', 'cloudcover_total', 'total_precipitation', 
            '10_metre_u_wind_component', '10_metre_v_wind_component',
            'direct_solar_radiation', 'diffuse_solar_radiation'
        ]
        historical_weather_cols =  [
            'temperature', 'surface_pressure',
            'cloudcover_total', 'total_precipitation',
            'windspeed_10m', 'winddirection_10m',
            'direct_solar_radiation', 'diffuse_radiation'
        ]

        df = (
            df
                .with_columns(
                    # Forward-fill 'electricity_price' with the last existing value
                    [pl.col('electricity_price')
                        .fill_null(strategy='forward')] + 
                    # Forward-fill 'gas_price' with the last existing value
                    [pl.col('gas_price')
                        .fill_null(strategy='forward')] +
                    # Forward-fill 'installed_capacity' with the last existing value for each prediction unit
                    [pl.col('installed_capacity')
                        .fill_null(strategy='forward')
                        .over(partition_by=(pl.col('prediction_unit_id'), pl.col('is_consumption')))] +
                    # Forward-fill forecast_weather columns with the last existing value for each prediction unit
                    [pl.col(f'fw_{col}')
                        .fill_null(strategy='forward')
                        .over(partition_by=(pl.col('prediction_unit_id'), pl.col('is_consumption'))) for col in forecast_weather_cols] +
                    # Forward-fill historical_weather columns with the last existing value for each prediction unit
                    [pl.col(f'hw_{col}')
                        .fill_null(strategy='forward')
                        .over(partition_by=(pl.col('prediction_unit_id'), pl.col('is_consumption'))) for col in historical_weather_cols]
                )
        )

        if self.mode != 'test':
            # Fill missing capacity_factor with the result of division of power by filled installed_capacity
            df = df.with_columns(
                    pl.when(pl.col('capacity_factor').is_null() & (pl.col('installed_capacity') > 0))  
                      .then(pl.col('power') / pl.col('installed_capacity'))  
                      .otherwise(pl.col('capacity_factor'))  
                      .alias('capacity_factor')  
                    )

        return df

    def create_lag_features(self, df: pl.DataFrame) -> pl.DataFrame:
        """Add lagged features"""

        lag_columns = [
            'capacity_factor', 'electricity_price', 'gas_price',
            'hw_temperature', 'hw_surface_pressure',
            'hw_cloudcover_total', 'hw_total_precipitation',
            'hw_windspeed_10m', 'hw_winddirection_10m',
            'hw_direct_solar_radiation', 'hw_diffuse_radiation',
        ]

        # Sort before filling missing values
        df = df.with_columns(pl.col("datetime").cast(pl.Datetime)).sort("datetime")

        if self.mode == 'train':  
            # Compute lag features for each column within each prediction_unit_id and is_consumption
            for col in lag_columns:
                for lag in self.n_lag:
                    # Use pl.col(col) to refer to the column inside the with_columns
                    df = df.with_columns([
                            pl.col(col).shift(lag).over(["prediction_unit_id", "is_consumption"])
                              .alias(f"{col}_lag_{lag}")
                            ])

        else:
            # Fill NaNs in lagged features with the result of calculation in retrain or test df
            df = df.with_columns([
                    pl.when(pl.col(col).is_null())  
                      .then(pl.col(col).shift(lag).over(["prediction_unit_id", "is_consumption"]))  
                      .otherwise(pl.col(col))  
                      .alias(f"{col}_lag_{lag}")  
                    for col in lag_columns for lag in self.n_lag  
                    ])

        return df
    
    def drop_rows(self, df: pl.DataFrame) -> pl.DataFrame:
        """Drop unnecessary rows after evaluating lagged features"""

        if self.mode == 'retrain' and self.min_retrain_datetime is not None:
            df = df.filter(pl.col('datetime') >= self.min_retrain_datetime)

        if self.mode == 'test' and self.min_test_datetime is not None:
            df = df.filter(pl.col('datetime') >= self.min_test_datetime)

        # Drop rows with NaNs (in lagged features)
        if self.mode != 'test':
            df = df.drop_nulls()
        
        return df
    
    def drop_features(self, df):
        """Drop power feature"""

        if self.mode != 'test':
            df.select(['row_id', 'power']).write_csv('power.csv')
        
        df = df.drop('power')

        return df
    
    def convert_pandas (self, df: pl.DataFrame) -> pd.DataFrame:
        """Convert to Pandas dataframe as CatBoost doesn’t natively support Polars"""

        df = df.to_pandas()

        return df
    
    def fit(self, X, y=None):
        """Fit method (required for sklearn pipeline compatibility)"""
        return self

    def transform(self, X):
        """Apply feature engineering pipeline"""

        if self.mode == 'train':
            df = self.add_datetime_features(X)
            df = self.handle_missing_values(df)
            df = self.create_lag_features(df)
            df = self.drop_rows(df)
            df = self.drop_features(df)
            df = self.convert_pandas(df)
        else:
            df = self.add_datetime_features(X)
            df = self.merge_data(df)
            df = self.handle_missing_values(df)
            df = self.create_lag_features(df)
            df = self.drop_rows(df)
            df = self.drop_features(df)
            df = self.convert_pandas(df)

        return df

    def fit_transform(self, X, y=None):
        self.fit(X)
        return self.transform(X)

In [9]:
# Train data pipeline instantiation
pipe_train = Pipeline([
    ('data_preparation', DataPreparation(mode='train')),
    ('data_merging', DataMerging(mode='train')),
    ('feature_engeneering', FeatureEngeneering(mode='train'))
])

In [10]:
# Run train data pipeline
train_df_model_11 = pipe_train.fit_transform(train_data)

In [11]:
train_df_model_11.to_csv('train_df_model_11.csv')

## Modelling

### Model Architecture 

Target: Capacity_factor <br>
Features: Exogeneous covariates, lagged power and other features, no decomposition <br>
Model: CatBoost with recursive forecasting (as forcasted target would be used as lagged power features) <br>
<br>
Pros: No data leakage <br>
Cons: Propagation of error due to recursive forecasting <br>

In [ ]:
# Define target and categorical features
target = 'capacity_factor'

cat_features = ['prediction_unit_id', 'is_consumption', 
                'county', 'is_business', 'product_type',  
                'is_holiday', 'is_day_before_holiday',
                'is_dst_spring_forward', 'is_dst_fall_backward'
]

In [ ]:
class CatBoostRegressor(BaseEstimator, TransformerMixin):

    def __init__(self, random_state, target, cat_features):
        """Initialize the CatBoostRegressor model"""

        self.random_state = random_state
        self.target = target
        self.cat_features = cat_features
        self.final_model_cb_consumption = None
        self.final_model_cb_production = None
        self.features = None

    def objective(self, trial, df, mask):
        """Optuna objective function for CatBoost hyperparameter optimization"""

        df_subset = df[mask]
        features = df_subset.columns.difference(['datetime', 'capacity_factor', 'row_id'])

        # TimeSeriesSplit for Cross-Validation
        tscv = TimeSeriesSplit(n_splits=3)

        params = {
            'iterations': trial.suggest_int('iterations', 100, 1000),
            'depth': trial.suggest_int('depth', 4, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.01, 10, log=True),
            'score_function': trial.suggest_categorical('score_function', ['Cosine', 'L2']),
            'loss_function': 'MAE',
            'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide'])
        }

        mae_scores = []
        for train_index, test_index in tscv.split(df_subset):
            train_df, test_df = df_subset.iloc[train_index], df_subset.iloc[test_index]
            X_train, y_train = train_df[features], train_df[self.target]
            X_test, y_test = test_df[features], test_df[self.target]
            
            # Prepare CatBoost data pools
            train_pool = Pool(X_train, y_train, cat_features=self.cat_features)
            test_pool = Pool(X_test, y_test, cat_features=self.cat_features)
            
            # Train CatBoost model
            model = CatBoost(**params)
            model.fit(train_pool, eval_set=test_pool, verbose=0)
            
            # Predict and evaluate performance
            predictions = model.predict(test_pool)
            mae = np.mean(np.abs(y_test - predictions))
            mae_scores.append(mae)

        return np.mean(mae_scores)

    def fit(self, df, y=None):
        """Fit the CatBoostRegressor model"""

        # Define features to use in the final model
        self.features = df.columns.difference(['datetime', 'capacity_factor', 'row_id'])

        # Create two separate Optuna studies for consumption and production
        study_cb_consumption = optuna.create_study(direction='minimize')
        study_cb_production = optuna.create_study(direction='minimize')

        # Run Optuna optimization for consumption subset
        self.mask_consumption = df['is_consumption'] == 1
        study_cb_consumption.optimize(lambda trial: self.objective(trial, df=df, mask=self.mask_consumption), n_trials=10)
        print("Best parameters for consumption:", study_cb_consumption.best_params)
        print("Best score for consumption:", study_cb_consumption.best_value)

        # Run Optuna optimization for production subset
        self.mask_production = df['is_consumption'] == 0
        study_cb_production.optimize(lambda trial: self.objective(trial, df=df, mask=self.mask_production), n_trials=10)
        print("Best parameters for production:", study_cb_production.best_params)
        print("Best score for production:", study_cb_production.best_value)

        # Train final models on the full dataset with the best parameters for each subset
        best_params_consumption = study_cb_consumption.best_params
        self.final_model_cb_consumption = CatBoost(**best_params_consumption, random_seed=self.random_state)
        final_pool_consumption = Pool(df[self.features][self.mask_consumption], df[self.target][self.mask_consumption], cat_features=self.cat_features)
        self.final_model_cb_consumption.fit(final_pool_consumption, verbose=100)

        best_params_production = study_cb_production.best_params
        self.final_model_cb_production = CatBoost(**best_params_production, random_seed=self.random_state)
        final_pool_production = Pool(df[self.features][self.mask_production], df[self.target][self.mask_production], cat_features=self.cat_features)
        self.final_model_cb_production.fit(final_pool_production, verbose=100)

        return self

    def transform(self, X, y=None):
        """Make predictions using the CatBoostRegressor model"""
        
        if self.final_model_cb_consumption is None or self.final_model_cb_production is None:
            raise RuntimeError("The model has not been fit yet. Please call `fit` before `transform`.")
        
        if self.features is None:
            raise RuntimeError("Features not defined. Please call `fit` or `load` with features parameter first.")
        
        # Make a copy of the test data to avoid modifying the original
        X_test = X.copy()
        
        # Add a predictions column to hold forecasted values
        X_test['predictions'] = np.nan
        
        # Sort data by datetime and prediction_unit_id to ensure sequential forecasting per unit
        X_test = X_test.sort_values(by=['datetime', 'prediction_unit_id', 'is_consumption']).reset_index(drop=True)
        
        # Define the lag values used in feature engineering (must match FeatureEngineering.n_lag)
        lag_values = [12, 24, 168, 720]
        
        # Pre-compute indices for consumption and production for batch processing
        consumption_mask = X_test['is_consumption'] == 1
        production_mask = X_test['is_consumption'] == 0
        
        n_rows = len(X_test)
        
        # Check if recursive forecasting is needed (i.e., if there are lag features that need updating)
        capacity_lag_cols = [f'capacity_factor_lag_{lag}' for lag in lag_values if f'capacity_factor_lag_{lag}' in X_test.columns]
        needs_recursive = len(capacity_lag_cols) > 0
        
        if needs_recursive:
            # Recursive forecasting needed - process row by row
            for idx in range(n_rows):
                # Select the features for the current row
                current_features = X_test[self.features].iloc[idx:idx+1]
                
                # Predict capacity factor based on 'is_consumption' status
                if X_test.at[idx, 'is_consumption'] == 1:
                    prediction = self.final_model_cb_consumption.predict(current_features)[0]
                else:
                    prediction = self.final_model_cb_production.predict(current_features)[0]
                
                # Store the prediction
                X_test.at[idx, 'predictions'] = prediction
                X_test.at[idx, 'capacity_factor'] = prediction
                
                # Recalculate lagged capacity_factor features for future rows
                for lag in lag_values:
                    future_idx = idx + lag
                    if future_idx < n_rows:
                        lag_column = f'capacity_factor_lag_{lag}'
                        if lag_column in X_test.columns:
                            # Only update if it's the same prediction unit
                            if (X_test.at[future_idx, 'prediction_unit_id'] == X_test.at[idx, 'prediction_unit_id'] and
                                X_test.at[future_idx, 'is_consumption'] == X_test.at[idx, 'is_consumption']):
                                X_test.at[future_idx, lag_column] = prediction
        else:
            # No recursive forecasting needed - batch predict for efficiency
            if consumption_mask.any():
                X_test.loc[consumption_mask, 'predictions'] = self.final_model_cb_consumption.predict(
                    X_test.loc[consumption_mask, self.features]
                )
            if production_mask.any():
                X_test.loc[production_mask, 'predictions'] = self.final_model_cb_production.predict(
                    X_test.loc[production_mask, self.features]
                )
        
        # Get installed_capacity
        installed_capacity = X_test['installed_capacity'].values
        
        # Calculate power as capacity_factor * installed_capacity
        X_test['target'] = X_test['predictions'] * installed_capacity

        # Ensure the target values are non-negative (MAE loss doesn't enforce positivity)
        X_test['target'] = np.maximum(X_test['target'], 0) 
        
        submission = X_test[['row_id', 'target']]
        
        return submission
    
    def compute_mae_power(self, model, df):
        """Compute train MAE for power using the trained models"""
    
        if model is None:
            raise RuntimeError("The models have not been fit yet")
        
        # Load actual power values
        revealed_power = pd.read_csv('power.csv')

        # Get predictions
        predicted_power = model.transform(df)

        # Merge actual and predicted values on row_id
        mae_calculation = pd.merge(
                                revealed_power[['row_id', 'power']], 
                                predicted_power[['row_id', 'target']], 
                                on='row_id', 
                                how='left'
                            )
        mae_calculation.to_csv('mae_calculation.csv')

        # Ensure there are no NaNs after merging
        mae_calculation = mae_calculation.dropna(subset=['power', 'target'])

        # Compute train MAE for power feature
        mae_power = np.mean(np.abs(mae_calculation['target'] - mae_calculation['power']))
    
        return mae_power

    def save(self, file_path):
        """Save the fitted model to a file"""

        if self.final_model_cb_consumption is not None and self.final_model_cb_production is not None:
            self.final_model_cb_consumption.save_model(f"{file_path}_model_11_consumption")
            self.final_model_cb_production.save_model(f"{file_path}_model_11_production")
            # Save features list for later loading
            import json
            with open(f"{file_path}_features.json", 'w') as f:
                json.dump(list(self.features), f)
            print(f"Model and features saved to {file_path}")
        else:
            raise RuntimeError("The model has not been fit yet. Cannot save an unfitted model.")

    def load(self, file_path_consumption, file_path_production, features=None, features_file=None):
        """Load a model from a file
        
        Args:
            file_path_consumption: Path to the consumption model file
            file_path_production: Path to the production model file
            features: List of feature names (optional if features_file provided)
            features_file: Path to JSON file with feature names (optional if features provided)
        """

        if os.path.exists(file_path_consumption) and os.path.exists(file_path_production):
            self.final_model_cb_consumption = CatBoost()
            self.final_model_cb_production = CatBoost()
            self.final_model_cb_consumption.load_model(file_path_consumption)
            self.final_model_cb_production.load_model(file_path_production)
            
            # Load features from file or use provided features
            if features is not None:
                self.features = pd.Index(features)
            elif features_file is not None and os.path.exists(features_file):
                import json
                with open(features_file, 'r') as f:
                    self.features = pd.Index(json.load(f))
            else:
                # Try to infer features from the model if possible
                try:
                    self.features = pd.Index(self.final_model_cb_consumption.feature_names_)
                except:
                    self.features = None
                    print("Warning: Features not set. Please provide features list or call set_features() before transform().")
            
            print("Model loaded")
        else:
            raise FileNotFoundError("No model found")
    
    def set_features(self, features):
        """Set features manually after loading"""
        self.features = pd.Index(features) if not isinstance(features, pd.Index) else features

In [ ]:
# Run the model
model_11 = CatBoostRegressor(random_state=RANDOM_STATE, target=target, cat_features=cat_features)
#model_11.fit(train_df_model_11)

# Load the previously saved model
model_11.load(file_path_consumption="/kaggle/input/catboost-consumption/other/default/1",
              file_path_production="/kaggle/input/catboost-production/other/default/1")

# Set features from training data (needed for transform)
model_11.set_features(train_df_model_11.columns.difference(['datetime', 'capacity_factor', 'row_id']))

In [ ]:
# Save the fitted model
#try:
#    model_11.save('catboost_model')
#    print("Model saved successfully.")
#except Exception as e:
#    print(f"Error saving the model: {e}")

In [ ]:
mae_train = model_11.compute_mae_power(model=model_11, df=train_df_model_11)
print(f"Train MAE for power: {mae_train}")

## Submission

In [16]:
try:
    import enefit
except ImportError as e:
    print(f"Error importing enefit: {e}")

In [17]:
env = enefit.make_env()
iter_test = env.iter_test()

In [ ]:
for (
        test, 
        revealed_targets, 
        client, 
        historical_weather, 
        forecast_weather, 
        electricity_prices, 
        gas_prices, 
        sample_prediction
    ) in iter_test:
         
        def convert_datetime(df, column):
            """Convert a column in a DataFrame to datetime with nanosecond (ns) precision."""
            df[column] = pd.to_numeric(df[column])
            for unit in [1_000_000_000, 1_000_000, 1_000]:
                try:
                    df[column] = pd.to_datetime(df[column] // unit, unit='s').astype('datetime64[us]')
                    break
                except pd.errors.OutOfBoundsDatetime:
                    continue
        
        # Convert date
        date_mappings = {
            'test': ['prediction_datetime'],
            'revealed_targets': ['datetime'],
            'client': ['date'],
            'electricity_prices': ['forecast_date'],
            'gas_prices': ['forecast_date'],
            'historical_weather': ['datetime'],
            'forecast_weather': ['forecast_datetime']
        }
        
        # Convert datetime columns
        for df_name, columns in date_mappings.items():
            df = globals()[df_name]  # Get the DataFrame object
            for col in columns:
                convert_datetime(df, col)
        
        # Convert datetime to date (truncate the time part)
        client['date'] = client['date'].dt.date
    
        #test.to_csv('test')
        #revealed_targets.to_csv('revealed_targets')
        #client.to_csv('client') 
        #historical_weather.to_csv('historical_weather')
        #forecast_weather.to_csv('forecast_weather') 
        #electricity_prices.to_csv('electricity_prices') 
        #gas_prices.to_csv('gas_prices')
    
        # Convert to polars dataframe
        test = pl.from_pandas(test)
        revealed_targets = pl.from_pandas(revealed_targets)
        client = pl.from_pandas(client) 
        historical_weather = pl.from_pandas(historical_weather)
        forecast_weather = pl.from_pandas(forecast_weather) 
        electricity_prices = pl.from_pandas(electricity_prices) 
        gas_prices = pl.from_pandas(gas_prices)
        
        # Initialize a retrain data dict
        retrain_data = {
            'power_df': revealed_targets,
            'client_df': client,
            'electricity_df': electricity_prices,
            'gas_df': gas_prices,
            'forecast_weather_df': forecast_weather,
            'historical_weather_df': historical_weather,
            'mapping_df': mapping_df
        }
            
        # Initialize a retrain data pipeline
        pipe_retrain = Pipeline([
            ('data_preparation', DataPreparation(mode='retrain')),
            ('data_merging', DataMerging(mode='retrain')),
            ('feature_engeneering', FeatureEngeneering(train_df=train_df_model_11, mode='retrain'))
        ])

        # Run the retrain data pipeline
        retrain_df_model_11 = pipe_retrain.fit_transform(retrain_data)

        # Save retrain data as csv
        retrain_df_model_11.to_csv('retrain_df_model_11.csv')

        # Initialize a test data dict
        test_data = {
            'power_df': test,
            'client_df': client,
            'electricity_df': electricity_prices,
            'gas_df': gas_prices,
            'forecast_weather_df': forecast_weather,
            'historical_weather_df': historical_weather,
            'mapping_df': mapping_df
        }
            
        # Initialize a test data pipeline
        pipe_test = Pipeline([
            ('data_preparation', DataPreparation(mode='test')),
            ('data_merging', DataMerging(mode='test')),
            ('feature_engeneering', FeatureEngeneering(train_df=train_df_model_11, retrain_df=retrain_df_model_11, mode='test'))
        ])

        # Run the test data pipeline
        test_df_model_11 = pipe_test.fit_transform(test_data)

        # Save test data as csv
        test_df_model_11.to_csv('test_df_model_11.csv')
          
        # Predict target
        sample_prediction = model_11.transform(test_df_model_11)
        sample_prediction['target'].fillna(0.0, inplace=True)
        sample_prediction['target'] = sample_prediction['target'].astype(float)
        
        # Submit 
        env.predict(sample_prediction)

This version of the API is not optimized and should not be used to estimate the runtime of your code on the hidden test set.
